In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/airlines-dataset/travel.sqlite


In [2]:
# Importing required libraries
import sqlite3
import numpy as np
import pandas as pd
import plotly.express as px
import folium
import pandasql as ps
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
from datetime import datetime

In [3]:
# Creating a SQL connection to SQLite database
con = sqlite3.connect("/kaggle/input/airlines-dataset/travel.sqlite")

In [4]:
# checking the tables in the database
query  = """
        SELECT name
        FROM sqlite_schema
        WHERE type ='table'
        """
table = pd.read_sql_query(query, con)
table

,name
0,aircrafts_data
1,airports_data
2,boarding_passes
3,bookings
4,flights
5,seats
6,ticket_flights
7,tickets


In [5]:
query  = """
        SELECT *
        FROM aircrafts_data 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,aircraft_code,model,range
0,773,"{""en"": ""Boeing 777-300"", ""ru"": ""Боинг 777-300""}",11100
1,763,"{""en"": ""Boeing 767-300"", ""ru"": ""Боинг 767-300""}",7900
2,SU9,"{""en"": ""Sukhoi Superjet-100"", ""ru"": ""Сухой Суп...",3000
3,320,"{""en"": ""Airbus A320-200"", ""ru"": ""Аэробус A320-...",5700
4,321,"{""en"": ""Airbus A321-200"", ""ru"": ""Аэробус A321-...",5600
5,319,"{""en"": ""Airbus A319-100"", ""ru"": ""Аэробус A319-...",6700
6,733,"{""en"": ""Boeing 737-300"", ""ru"": ""Боинг 737-300""}",4200
7,CN1,"{""en"": ""Cessna 208 Caravan"", ""ru"": ""Сессна 208...",1200
8,CR2,"{""en"": ""Bombardier CRJ-200"", ""ru"": ""Бомбардье ...",2700


In [6]:
query = """
    SELECT aircraft_code, json_extract(model, '$.en') AS aircraft_model, range
    FROM aircrafts_data
    LIMIT 10
"""
table = pd.read_sql_query(query, con)
table

,aircraft_code,aircraft_model,range
0,773,Boeing 777-300,11100
1,763,Boeing 767-300,7900
2,SU9,Sukhoi Superjet-100,3000
3,320,Airbus A320-200,5700
4,321,Airbus A321-200,5600
5,319,Airbus A319-100,6700
6,733,Boeing 737-300,4200
7,CN1,Cessna 208 Caravan,1200
8,CR2,Bombardier CRJ-200,2700


In [7]:
query  = """
        SELECT airport_code, json_extract(airport_name, '$.en') AS airport_name, 
        json_extract(city, '$.en') AS city, coordinates, timezone
        FROM airports_data 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,airport_code,airport_name,city,coordinates,timezone
0,YKS,Yakutsk Airport,Yakutsk,"(129.77099609375,62.0932998657226562)",Asia/Yakutsk
1,MJZ,Mirny Airport,Mirnyj,"(114.03900146484375,62.534698486328125)",Asia/Yakutsk
2,KHV,Khabarovsk-Novy Airport,Khabarovsk,"(135.18800354004,48.5279998779300001)",Asia/Vladivostok
3,PKC,Yelizovo Airport,Petropavlovsk,"(158.453994750976562,53.1679000854492188)",Asia/Kamchatka
4,UUS,Yuzhno-Sakhalinsk Airport,Yuzhno-Sakhalinsk,"(142.718002319335938,46.8886985778808594)",Asia/Sakhalin
5,VVO,Vladivostok International Airport,Vladivostok,"(132.147994995117188,43.3989982604980469)",Asia/Vladivostok
6,LED,Pulkovo Airport,St. Petersburg,"(30.2625007629394531,59.8003005981445312)",Europe/Moscow
7,KGD,Khrabrovo Airport,Kaliningrad,"(20.5925998687744141,54.8899993896484375)",Europe/Kaliningrad
8,KEJ,Kemerovo Airport,Kemorovo,"(86.1072006225585938,55.2700996398925781)",Asia/Novokuznetsk
9,CEK,Chelyabinsk Balandino Airport,Chelyabinsk,"(61.503300000000003,55.3058010000000024)",Asia/Yekaterinburg


In [8]:
query  = """
        SELECT *
        FROM boarding_passes 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,ticket_no,flight_id,boarding_no,seat_no
0,0005435212351,30625,1,2D
1,0005435212386,30625,2,3G
2,0005435212381,30625,3,4H
3,0005432211370,30625,4,5D
4,0005435212357,30625,5,11A
5,0005435212360,30625,6,11E
6,0005435212393,30625,7,11H
7,0005435212374,30625,8,12E
8,0005435212365,30625,9,13D
9,0005435212378,30625,10,14H


In [9]:
query  = """
        SELECT *
        FROM bookings 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,book_ref,book_date,total_amount
0,00000F,2017-07-05 03:12:00+03,265700
1,000012,2017-07-14 09:02:00+03,37900
2,000068,2017-08-15 14:27:00+03,18100
3,000181,2017-08-10 13:28:00+03,131800
4,0002D8,2017-08-07 21:40:00+03,23600
5,0002DB,2017-07-29 06:30:00+03,101500
6,0002E0,2017-07-11 16:09:00+03,89600
7,0002F3,2017-07-10 05:31:00+03,69600
8,00034E,2017-08-04 16:52:00+03,73300
9,000352,2017-07-06 02:02:00+03,109500


In [10]:
query  = """
        SELECT *
        FROM flights 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,flight_id,flight_no,scheduled_departure,scheduled_arrival,departure_airport,arrival_airport,status,aircraft_code,actual_departure,actual_arrival
0,1185,PG0134,2017-09-10 09:50:00+03,2017-09-10 14:55:00+03,DME,BTK,Scheduled,319,\N,\N
1,3979,PG0052,2017-08-25 14:50:00+03,2017-08-25 17:35:00+03,VKO,HMA,Scheduled,CR2,\N,\N
2,4739,PG0561,2017-09-05 12:30:00+03,2017-09-05 14:15:00+03,VKO,AER,Scheduled,763,\N,\N
3,5502,PG0529,2017-09-12 09:50:00+03,2017-09-12 11:20:00+03,SVO,UFA,Scheduled,763,\N,\N
4,6938,PG0461,2017-09-04 12:25:00+03,2017-09-04 13:20:00+03,SVO,ULV,Scheduled,SU9,\N,\N
5,7784,PG0667,2017-09-10 15:00:00+03,2017-09-10 17:30:00+03,SVO,KRO,Scheduled,CR2,\N,\N
6,9478,PG0360,2017-08-28 09:00:00+03,2017-08-28 11:35:00+03,LED,REN,Scheduled,CR2,\N,\N
7,11085,PG0569,2017-08-24 15:05:00+03,2017-08-24 16:10:00+03,SVX,SCW,Scheduled,733,\N,\N
8,11847,PG0498,2017-09-12 10:15:00+03,2017-09-12 14:55:00+03,KZN,IKT,Scheduled,319,\N,\N
9,12012,PG0621,2017-08-26 16:05:00+03,2017-08-26 17:00:00+03,KZN,MQF,Scheduled,CR2,\N,\N


In [11]:
query  = """
        SELECT *
        FROM seats 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,aircraft_code,seat_no,fare_conditions
0,319,2A,Business
1,319,2C,Business
2,319,2D,Business
3,319,2F,Business
4,319,3A,Business
5,319,3C,Business
6,319,3D,Business
7,319,3F,Business
8,319,4A,Business
9,319,4C,Business


In [12]:
query  = """
        SELECT *
        FROM ticket_flights 
        LIMIT 10
        """
table = pd.read_sql_query(query, con)
table

,ticket_no,flight_id,fare_conditions,amount
0,0005432159776,30625,Business,42100
1,0005435212351,30625,Business,42100
2,0005435212386,30625,Business,42100
3,0005435212381,30625,Business,42100
4,0005432211370,30625,Business,42100
5,0005435212357,30625,Comfort,23900
6,0005435212360,30625,Comfort,23900
7,0005435212393,30625,Comfort,23900
8,0005435212374,30625,Comfort,23900
9,0005435212365,30625,Comfort,23900


In [13]:
query  = """
        SELECT *
        FROM tickets
        LIMIT 10;
        """
table = pd.read_sql_query(query, con)
table

,ticket_no,book_ref,passenger_id
0,0005432000987,06B046,8149 604011
1,0005432000988,06B046,8499 420203
2,0005432000989,E170C3,1011 752484
3,0005432000990,E170C3,4849 400049
4,0005432000991,F313DD,6615 976589
5,0005432000992,F313DD,2021 652719
6,0005432000993,F313DD,0817 363231
7,0005432000994,CCC5CB,2883 989356
8,0005432000995,CCC5CB,3097 995546
9,0005432000996,1FB1E4,6866 920231


Customer loyalty: Find passengers who travel frequently (by counting tickets per passenger).

In [14]:
query='''
    SELECT t.ticket_no, COUNT(*) AS numberOfticket	
    FROM tickets t
    INNER JOIN ticket_flights f
    ON t.ticket_no=f.ticket_no
    GROUP BY t.ticket_no
    ORDER BY numberOfticket DESC
    LIMIT 10;
    '''
df=pd.read_sql_query(query, con)
df

,ticket_no,numberOfticket
0,0005435983728,6
1,0005435983727,6
2,0005435787413,6
3,0005435787412,6
4,0005435787411,6
5,0005435787410,6
6,0005435787409,6
7,0005435787408,6
8,0005435787407,6
9,0005435787406,6
